# Accuracy heatmap and error vs timing noise — Plots 3 and 10

How accurate is a position fix **at each spot on the floor** for a given sensor layout? Use this to choose where to tape the triangle before moving real sensors.

* **Fast method:** at each point, the geometry (Jacobian) gives the expected error: `cov = σ_t² c² (JᵀJ)⁻¹`. Cheap, so we map the whole floor.
* **Check:** at coarse points we simulate 50 knocks with timing noise, solve each, and take the median error. If the two agree, the map can be trusted.
* **Air vs water** at the same sampling rate: sound in water is ~4.3× faster, so every µs of timing error costs ~4.3× more distance.

In [ ]:
# ===================== EDIT THESE =====================
LAYOUTS = {                                   # name: sensor (x, y) in meters. First one = "current".
    "2 m triangle (current)": [(0, 0), (2, 0), (1, 1.732)],
    "3 m triangle":           [(0, 0), (3, 0), (1.5, 2.598)],
}
SIGMA_T_S = 100e-6        # timing noise per sensor (replace with the jitter measured in 02_calibration)
FS_HZ = 10_000            # only used to mark "one sample" on Plot 10
TARGET_M = 0.15           # accuracy target from the M0 doc
MARGIN_M = 1.5            # how far outside the sensors to map
# ======================================================

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, matplotlib.pyplot as plt
import accuracy as A
from arraycfg import speed_of_sound
C_AIR, C_WATER = speed_of_sound("air"), speed_of_sound("water")
F = A.median_from_cov_factor()                 # RMS radial -> median radial
name0 = next(iter(LAYOUTS)); S = np.array(LAYOUTS[name0], float)
names = [f"S{i+1}" for i in range(len(S))]
print(f"c air {C_AIR:.1f} m/s, c water {C_WATER:.1f} m/s, sigma_t {SIGMA_T_S*1e6:.0f} us")

## Plot 3: expected error across the floor, air vs water (current layout)

In [ ]:
xs, ys = A.grid_for(S, MARGIN_M, 0.05)
m_air = A.crlb_map(S, xs, ys, C_AIR, SIGMA_T_S) * F
m_wat = A.crlb_map(S, xs, ys, C_WATER, SIGMA_T_S) * F
sa = A.summarize_map(m_air / F, xs, ys, S, TARGET_M); sw = A.summarize_map(m_wat / F, xs, ys, S, TARGET_M)
title = (f"{name0}: {sa['inside_median_m']*100:.1f} cm inside the triangle in air, but {sa['outside_median_m']*100:.0f} cm "
         f"just outside; in water {sw['inside_median_m']*100:.0f} cm inside (misses the {TARGET_M*100:.0f} cm target)")
fig = A.plot_heatmaps([m_air, m_wat], S, names, xs, ys,
                      [f"air, c = {C_AIR:.0f} m/s", f"water, c = {C_WATER:.0f} m/s (same {FS_HZ/1000:.0f} kHz, same σ_t)"], title)
fig.savefig(ROOT / "docs/figures/plot3_accuracy_heatmap.png", dpi=130); plt.show()
for k, v in sa.items(): print(f"air   {k}: {v}")

### Cross-check: does the fast formula match brute-force simulation?

In [ ]:
cx, cy = A.grid_for(S, MARGIN_M, 0.5)
mc = A.mc_map(S, cx, cy, C_AIR, SIGMA_T_S, n=50)
cr = A.crlb_map(S, cx, cy, C_AIR, SIGMA_T_S) * F
X, Y = np.meshgrid(cx, cy); ins = A.inside_hull(S, X, Y)
ratio = mc / cr
print(f"inside the triangle: simulation / formula = {np.median(ratio[ins]):.2f} (range {ratio[ins].min():.2f}-{ratio[ins].max():.2f})")
print(f"outside: median ratio {np.median(ratio[~ins]):.2f}; {int((mc[~ins] >= A.LOST_M * 0.5).sum())} of {(~ins).sum()} points "
      "have half their knocks solved to the wrong spot (3 sensors can give two valid answers outside the triangle)")
verdict = "agree" if 0.8 <= np.median(ratio[ins]) <= 1.25 else "DISAGREE"
print(f"VERDICT: the two methods {verdict} inside the triangle; outside, trust the simulation, not the formula.")

## Compare layouts before moving sensors

In [ ]:
rows = []
maps, titles = [], []
for name, pts in LAYOUTS.items():
    P = np.array(pts, float); gx, gy = A.grid_for(P, MARGIN_M, 0.05)
    m = A.crlb_map(P, gx, gy, C_AIR, SIGMA_T_S) * F
    s = A.summarize_map(m / F, gx, gy, P, TARGET_M)
    rows.append((name, s["area_under_target_m2"], s["inside_median_m"] * 100))
    maps.append((m, P, gx, gy)); titles.append(name)
for name, area, med in rows:
    print(f"{name:28s} area under {TARGET_M*100:.0f} cm: {area:5.1f} m²   median inside: {med:.1f} cm")

## Plot 10: error vs timing noise, air vs water

In [ ]:
J = np.logspace(-5, -3, 7)                    # 10 us .. 1 ms
med_air = A.error_vs_jitter(S, C_AIR, J, n=150)
med_wat = A.error_vs_jitter(S, C_WATER, J, n=150)
slope = np.polyfit(np.log10(J[:5]), np.log10(med_air[:5]), 1)[0]
max_air = np.interp(np.log10(TARGET_M), np.log10(med_air), np.log10(J)); max_wat = np.interp(np.log10(TARGET_M), np.log10(med_wat), np.log10(J))
title = (f"Error grows {10**slope:.0f}× per 10× timing noise; {TARGET_M*100:.0f} cm needs σ_t ≤ {10**max_air*1e6:.0f} µs in air "
         f"but ≤ {10**max_wat*1e6:.0f} µs in water")
fig = A.plot_error_vs_jitter(J, {"air": (med_air, style_air := "#0072B2"), "water": (med_wat, "#E69F00")}, 1 / FS_HZ, title, TARGET_M)
fig.savefig(ROOT / "docs/figures/plot10_error_vs_jitter.png", dpi=130); plt.show()

## Recommendation

(Written from the numbers above; rerun after changing `LAYOUTS` or `SIGMA_T_S`.)

In [ ]:
a0, a1 = rows[0][1], rows[-1][1]
print(f"* Inside the {name0} the error is ~{sa['inside_median_m']*100:.0f} cm, well under target. Outside it, error climbs "
      f"fast (~{sa['outside_median_m']*100:.0f} cm within {MARGIN_M} m) and some knocks solve to a mirror spot: "
      "that is the dead zone.")
print(f"* So put the triangle AROUND the search area, not beside it. If the area is bigger than the triangle, "
      f"spread the sensors: '{rows[-1][0]}' covers {a1:.1f} m² under {TARGET_M*100:.0f} cm vs {a0:.1f} m² now "
      f"(same accuracy inside), as long as every sensor still hears a knock clearly.")
print(f"* Underwater needs ~{C_WATER/C_AIR:.1f}x tighter timing for the same accuracy: faster sampling plus sub-sample onsets, "
      "not just waterproof sensors.")